# 08 F1 dose–response: figures from the analysis tables

Every figure is drawn by `silent_rag.plots` from the tables in `results/analysis_f1_v3/`, written by
`silent_rag.analyze_benchmark_results` (manifest: `experiments/analysis_f1/manifests/analysis_f1_v3_2026-11-18.json`).
This notebook computes nothing. Style: `experiments/analysis_f1/figures.yaml`. Figures are saved to `results/figures/analysis_f1_v3/`.

In [ ]:
%matplotlib inline
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')  # macOS: avoids an OpenMP conflict between faiss and torch
os.environ.setdefault('OMP_NUM_THREADS', '1')

from pathlib import Path

import pandas as pd
from IPython.display import display

# run from the repo root, so experiments/ and results/ resolve
ROOT = Path.cwd() if (Path.cwd() / 'configs').exists() else Path.cwd().parent
os.chdir(ROOT)

from silent_rag.plots import figures_models as fm
from silent_rag.plots import figures_statistics as fs
from silent_rag.plots.style import FIGURE_DEFAULTS, PlotStyle

style = PlotStyle.from_yaml('experiments/analysis_f1/figures.yaml')
style.check_figure_names(FIGURE_DEFAULTS)  # every figures: entry names a built figure
TABLES, OUT = Path('results/analysis_f1_v3'), Path('results/figures/analysis_f1_v3')
OUT.mkdir(parents=True, exist_ok=True)

models = pd.read_parquet(TABLES / 'replacement_model_difference_table.parquet')  # table 16: label order in every figure
drops = pd.read_parquet(TABLES / 'mean_ndcg10_drop_all_test_questions_by_severity.parquet')  # table 9
shares = pd.read_parquet(TABLES / 'share_of_windows_with_significant_ndcg10_drop_by_severity.parquet')  # table 2
seconds = pd.read_parquet(TABLES / 'median_seconds_per_statistic_evaluation.parquet')  # table 3
auroc = pd.read_parquet(TABLES / 'auroc_statistic_harmful_vs_negative_windows_by_severity.parquet')  # table 4
auroc_by_drop = pd.read_parquet(TABLES / 'auroc_statistic_harmful_vs_negative_windows_by_min_ndcg10_drop.parquet')  # table 5
spearman = pd.read_parquet(TABLES / 'spearman_statistic_vs_window_mean_ndcg10_drop.parquet')  # table 6
stats = pd.read_parquet(TABLES / 'statistic_all_test_questions_vs_validation_reference_by_severity.parquet')  # table 10
STATISTICS = list(seconds['statistic'])  # every monitoring statistic in the run, cheapest first
STATISTICS

## Replacement models vs the baseline

Top-10 documents shared with the baseline, then nDCG@10 of the fully switched system, against the mean cosine of each
replacement model's document vectors with the baseline's (table 16); error bars are 95% CIs over questions.

In [ ]:
for y in ('shared', 'switched_ndcg10'):
    fig = fm.replacement_model_difference(models, y=y, style=style)
    fig.savefig(OUT / f'replacement_model_difference_{y}.png', dpi=style.resolve('replacement_model_difference')['figure']['dpi'])
    display(fig)

## Dose–response

Mean nDCG@10 drop over all test questions vs severity, as a drop, as nDCG@10 and as a percent of the clean baseline (table 9);
the band is the 95% CI over questions.

In [ ]:
for scale in ('drop', 'ndcg', 'percent_of_baseline'):
    fig = fm.dose_response_ndcg10_drop(drops, scale=scale, models=models, style=style)
    fig.savefig(OUT / f'dose_response_ndcg10_drop_{scale}.png', dpi=style.resolve('dose_response_ndcg10_drop')['figure']['dpi'])
    display(fig)

## Drop at full severity vs mean cosine

Mean nDCG@10 drop over all test questions at severity 1 against each replacement model's mean cosine (tables 9 and 16);
error bars are 95% CIs over questions.

In [ ]:
fig = fm.ndcg10_drop_at_severity_vs_mean_cosine(drops, models, style=style)
fig.savefig(OUT / 'ndcg10_drop_at_severity_vs_mean_cosine.png', dpi=style.resolve('ndcg10_drop_at_severity_vs_mean_cosine')['figure']['dpi'])
display(fig)

## Windows with a significant drop

Percent of 100-question windows with a significant mean nDCG@10 drop vs severity, at minimum mean drops 0, 0.02 and 0.06
(table 2); shares of all windows, no interval.

In [ ]:
for min_drop in (0.0, 0.02, 0.06):
    fig = fs.share_of_significant_windows_vs_severity(shares, min_drop=min_drop, models=models, style=style)
    fig.savefig(OUT / f'share_of_significant_windows_vs_severity_{min_drop:g}.png',
                dpi=style.resolve('share_of_significant_windows_vs_severity')['figure']['dpi'])
    display(fig)

## Seconds per evaluation

Median seconds per evaluation of each monitoring statistic (table 3); medians over all evaluations, no interval.

In [ ]:
fig = fs.seconds_per_statistic(seconds, style=style)
fig.savefig(OUT / 'seconds_per_statistic.png', dpi=style.resolve('seconds_per_statistic')['figure']['dpi'])
display(fig)

## Each statistic on all test questions

Each monitoring statistic computed on all test questions against the validation reference, vs severity (table 10);
the band is the range over seeds, not a confidence interval.

In [ ]:
for statistic in STATISTICS:
    fig = fs.statistic_vs_severity_all_test_questions(stats, statistic=statistic, models=models, style=style)
    fig.savefig(OUT / f'statistic_vs_severity_all_test_questions_{statistic}.png',
                dpi=style.resolve('statistic_vs_severity_all_test_questions')['figure']['dpi'])
    display(fig)

## AUROC by severity

AUROC of each statistic, windows with a significant drop vs unmodified windows, vs severity at minimum mean drop 0 (table 4);
the band is the 95% CI over random seeds (whole seeds resampled).

In [ ]:
for statistic in STATISTICS:
    fig = fs.auroc_vs_severity(auroc, statistic=statistic, min_drop=0.0, models=models, style=style)
    fig.savefig(OUT / f'auroc_vs_severity_{statistic}.png', dpi=style.resolve('auroc_vs_severity')['figure']['dpi'])
    display(fig)

## AUROC by minimum drop

AUROC of each statistic vs the minimum mean drop that makes a window significant, all severities pooled (table 5);
the band is the 95% CI over random seeds (whole seeds resampled).

In [ ]:
for statistic in STATISTICS:
    fig = fs.auroc_vs_min_ndcg10_drop(auroc_by_drop, statistic=statistic, models=models, style=style)
    fig.savefig(OUT / f'auroc_vs_min_ndcg10_drop_{statistic}.png', dpi=style.resolve('auroc_vs_min_ndcg10_drop')['figure']['dpi'])
    display(fig)

## Spearman correlation with the window's drop

Spearman ρ between each statistic and the window's mean nDCG@10 drop, per severity and with all severities pooled (table 6);
band and error bars are 95% CIs over random seeds (whole seeds resampled).

In [ ]:
for statistic in STATISTICS:
    fig = fs.spearman_vs_severity(spearman, statistic=statistic, models=models, style=style)
    fig.savefig(OUT / f'spearman_vs_severity_{statistic}.png', dpi=style.resolve('spearman_vs_severity')['figure']['dpi'])
    display(fig)